# P42 — Explicar y aprovechar los ejemplos adversarios

## 1. Título y paper

**Paper:** *Explaining and Harnessing Adversarial Examples*  
**Autoría:** Ian J. Goodfellow, Jonathon Shlens, Christian Szegedy  
**Año y venue:** 2014 · arXiv:1412.6572 · ICLR 2015  
**Nivel:** L3 · **Motor:** `adversarial`  
**Ficha completa:** [`P42_adversarial`](../../papers/foundational/P42_adversarial/README.md)

**Hito:** Una perturbación imperceptible cambia la predicción. Y la causa no es la profundidad: es la linealidad en dimensión alta.

- [arXiv:1412.6572](https://arxiv.org/abs/1412.6572)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Szegedy et al. (2013) habían descubierto que perturbaciones minúsculas engañaban a las redes, y se atribuía a la extrema no linealidad de los modelos profundos.
2. Ejecutar una implementación mínima de la propuesta: Mostrar que la explicación es la contraria —el comportamiento demasiado LINEAL en alta dimensión— y derivar de ahí un ataque de un solo paso (FGSM) y una defensa por entrenamiento adversario.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P04


## 4. Intuición

Cambia cada píxel de una foto en una milésima —invisible— pero **todos en la dirección que más confunde al modelo**. En una imagen con miles de píxeles, esas milésimas suman lo suficiente para cambiar la predicción.


## 5. Concepto mínimo

```text
FGSM:  x' = x + ε · sign(∇ₓ L(θ, x, y))

Para un modelo lineal wᵀx:
    cambio = wᵀ(x' − x) = ε · Σᵢ |wᵢ|      ← crece con la DIMENSIÓN
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('adversarial', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. Con ε=0,01, ¿cuánto cambia la salida en dimensión 10? ¿Y en 10 000?
2. ¿Es un problema de la profundidad de la red?
3. ¿Por qué la perturbación es imperceptible y el efecto no?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('adversarial', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('adversarial', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El cambio crece proporcionalmente a la dimensión. Con 10 000 componentes y ε=0,01, la salida se mueve ~100 unidades. **La causa es la linealidad en alta dimensión**, no la complejidad del modelo — que es justo lo contrario de lo que se creía.


## 10. Comentario pedagógico

El resultado más inquietante del paper no está en esta miniatura: los ejemplos adversarios **transfieren** entre modelos distintos entrenados con datos distintos. Eso significa que no son un bug de un modelo, sino una propiedad de los datos y de la clase de funciones.


## 11. Error o anti-patrón deliberado

Anti-patrón: creer que un modelo con alta exactitud es un modelo robusto.


In [ ]:
print('Exactitud 99% en el conjunto de test ← distribucion natural')
print('Exactitud  0% bajo perturbacion adversaria ← distribucion elegida por un atacante')
print('Son dos metricas distintas, y la segunda casi nunca se reporta.')

## 12. Corrección

Lo que hay que medir si el modelo se despliega donde alguien puede atacarlo:


In [ ]:
evaluacion = {'exactitud_limpia': 'sobre el test estandar',
              'exactitud_robusta': 'bajo ataque, con epsilon declarado',
              'ataque_usado': 'FGSM, PGD, adaptativo... y sus parametros',
              'transferencia': 'si el ataque generado en otro modelo tambien funciona'}
show(evaluacion)

## 13. Desafío guiado

Calcula qué ε hace falta en dimensión 784 (una imagen 28×28) para mover la salida 10 unidades.


In [ ]:
r = run_paper_lab('adversarial', seed=3)['result']
show(r)

## 14. Desafío autónomo

Implementa FGSM sobre un clasificador pequeño y mide la exactitud en función de ε. Después prueba entrenamiento adversario y comprueba cuánta exactitud limpia cuesta la robustez.


## 15. Evidencia de aprendizaje

Guarda la tabla de cambio frente a dimensión, y tu protocolo de evaluación con exactitud limpia y robusta por separado.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P42_adversarial/README.md) · evaluación formal: [`assessments/papers/P42_adversarial.md`](../../assessments/papers/P42_adversarial.md)


## 16. Cierre

La robustez es un problema abierto. Volvamos al entrenamiento: qué hace que una red profunda sea entrenable.


## 17. Conexión con el siguiente hito

- seguridad de modelos
- P52

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
